## *Installing required packages*

In [1]:
!pip install -U peft transformers bitsandbytes accelerate trl datasets

## Importing the libaries

In [2]:
import torch
from transformers import(AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, pipeline, HfArgumentParser, logging) #why using each import?
from peft import LoraConfig, PeftModel
from datasets import load_dataset
from trl import SFTTrainer,SFTConfig

In case of LLama model we need a prompt template for finetuning chat models



```
# System prompt - to guide the model
User Prompt - to give instruction
Model Answer
```





Generic prompt format for llama chat model

```
# <s>[INST]<<SYS>>
System prompt
<</SYS>>
User Prompt [/INST] </s>
```



we need to reformat our dataset to follow this template
Using this dataset - https://huggingface.co/datasets/timdettmers/openassistant-guanaco repo from hf



note: we dont have to follow a prompt format because we are using a llama base model, instead of chat model

In [4]:
#Loading and Formatting dataset

from datasets import load_dataset
import re

dataset = load_dataset("timdettmers/openassistant-guanaco")

#loading small number of rows since we are running on T4 GPU
dataset = dataset['train'].select(range(1000))

#lets define a function to transform the data
def transform_dataset(example):
  conversation_text = example['text']
  segments = conversation_text.split("###")

  reformatted_segments = []

#since the dataset already has human and assistant pairs (acting as system and user prompts to fit our template), lets use regex to seperate and format them.

  for i in range(1,len(segments)-1, 2):
    human_text = segments[i].strip().replace('Human:', '').strip()
    if i+1 <len(segments):
      bot_text = segments[i+1].strip().replace('Assistant:', '').strip()

      reformatted_segments.append(f'<s>[INST] {human_text}[/INST] {bot_text}</s>')
    else :
      reformatted_segments.append(f'<s>[INST] {human_text}[/INST]</s>')

  return {'text': reformatted_segments}
mapped_dataset = dataset.map(transform_dataset)
flattened_dataset = mapped_dataset.map(
    lambda batch: {"text": [entry for sublist in batch["text"] for entry in sublist]},
    batched=True,
    remove_columns=mapped_dataset.column_names,
)
#print(flattened_dataset[0])

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Repo card metadata block was not found. Setting CardData to empty.


##  Finetuning Setup

In [5]:
#load the model and train it on given 1000 samples.
model_name = "NousResearch/Hermes-3-Llama-3.1-8B"

# Fine-tuned result model name
new_model = "Llama-3.1-8b-chat-finetune"

In [6]:
# bitsandbytes parameters
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)
#loading model and tokenizer
model = AutoModelForCausalLM.from_pretrained(model_name,quantization_config = bnb_config, device_map = "auto")
model.config.use_cache = False
model.config.pretraining_tp = 1

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

config.json:   0%|          | 0.00/883 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

In [7]:
# Load LoRA configuration
peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.1,
    r=64,
    bias="none",
    task_type="CAUSAL_LM",
)

In [8]:
# TrainingArguments parameters, as SFT Config
sft_config = SFTConfig(
    output_dir="./results",
    num_train_epochs=1,
    fp16=False,
    bf16=False,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=2,
    gradient_checkpointing=True,
    max_grad_norm=0.3,
    learning_rate=2e-4,
    weight_decay=0.001,
    optim="paged_adamw_32bit",
    lr_scheduler_type="cosine",
    max_steps=-1,
    report_to="none",
    warmup_ratio=0.03,
    group_by_length=True,
    save_steps=0,
    logging_steps=25,
    max_seq_length=128,
    packing=False,
    dataset_text_field="text"
)

In [9]:
#Setting up SFTtrainer and kicking of training
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=flattened_dataset,
    peft_config=peft_config,
    args=sft_config
)

trainer.train()

Adding EOS to train dataset:   0%|          | 0/1263 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1263 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1263 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Step,Training Loss
25,1.966100
50,1.965400
75,1.568100
100,1.749300
125,1.564900
150,1.687600
175,1.588600
200,1.641300
225,1.487000
250,1.712700


TrainOutput(global_step=316, training_loss=1.6731405197819578, metrics={'train_runtime': 2490.5037, 'train_samples_per_second': 0.507, 'train_steps_per_second': 0.127, 'total_flos': 6644609719934976.0, 'train_loss': 1.6731405197819578})

## Saving model locally, uncomment for uploading to HF

In [10]:
#from huggingface_hub import login, create_repo, upload_folder

#login(token="your_hf_token")

#comment if you want to save only on HF
save_path = "./lora_finetuned_model"
trainer.model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

# repo_name = "your-username/Llama-3.1-8b-chat-finetune"
# create_repo(repo_name, exist_ok=True)

# upload_folder(
#     repo_id=repo_name,
#     folder_path=save_path,
#     path_in_repo=".",
#     commit_message="Upload LoRA-finetuned model"
# )


('./lora_finetuned_model/tokenizer_config.json',
 './lora_finetuned_model/special_tokens_map.json',
 './lora_finetuned_model/chat_template.jinja',
 './lora_finetuned_model/additional_chat_templates/tool_use.jinja',
 './lora_finetuned_model/tokenizer.json')